# Data Preparationfor Fine-Tuning

**Module 9 · Lesson 9.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Three Data Formats
- Chat Templates - The Silent Killer
- Quality Filtering - Drop Garbage Before Training
- Quality Crushes Quantity - The AlpaGasus Filter
- Synthetic Data Generation - and Always Filter It
- The Cleaning Pipeline - Cheapest Filters First

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The Fine-Tune That Threw No Error

> 💡 **Info**
>
> A team had 20,000 clean, hand-checked examples in **Alpaca format** (`instruction / input / output`). They pointed a fine-tune at a **Llama 3** chat model, hit run, and it trained to completion - **no error, no warning**. The loss curve looked normal. Then the model shipped and rambled: it never stopped cleanly, ignored the system prompt, and mixed up turns. The bug was invisible because it was never an *exception* - the data was fed to the model **wrapped in the wrong delimiters**. Llama 3 was pre-trained to see `<|start_header_id|>` and `<|eot_id|>` around each turn; the Alpaca text had none of them, so the model trained on a format it had never learned. **A chat-template mismatch is the single most common silent failure in fine-tuning - and avoiding it is what most of this lesson is really about.**

### 🎯 What you will be able to do after this lesson

- **Build** the three fine-tuning data formats (instruction, conversation, preference) and export them as JSONL.

- **Operate** `apply_chat_template()` correctly - the #1 silent failure - with the special-token and PAD rules that go with it.

- **Compare** data by QUALITY not quantity: LLM-as-judge scoring, the AlpaGasus filter, and a cheapest-first cleaning pipeline (dedup / contamination / PII).

- **Evaluate** a dataset for production: stratified splits, versioning, and the India path (tokenizer fertility, NFC, DPDP).

> 📦 **Info**
>
> ✅ Before you startThis is the hands-on follow-on to **Lesson 9.1**, which kept saying data curation is ~80% of fine-tuning. Here you build that data. You should be comfortable with the fine-tuning decision (9.1), JSON, and basic Python. The LoRA/QLoRA training that consumes this data is **Lesson 9.5**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🍳 **Analogy**
>
> **Fine-tuning data is ingredients for a restaurant.** Bad ingredients make bad food no matter how skilled the chef, so you need the right FORMAT (recipe structure), QUALITY filtering (throw out the rotten ones), and enough - but not endless - QUANTITY (100 perfect examples beat 10,000 messy ones). *Where the analogy breaks down:* even perfect ingredients plated on the wrong dish ruin the meal - that is the **chat template**. The wrapper the model expects around each turn matters as much as what is inside it.

> 📦 **Info**
>
> ⚠️ The misconception that quietly ruins runs**“Any reasonable data format works - the model adapts.”** It does not. Every model was pre-trained on a SPECIFIC token format with special delimiters. Feed it a different wrapper and it trains on text it has never seen - with **no exception thrown**. The result is a quietly degraded model that is very hard to debug. The only safe formatter is the model’s own `apply_chat_template()`.

> 💡 **Info**
>
> 🚫 Anti-pattern: collect more instead of filteringWhen a fine-tune underperforms, the instinct is “get more data.” Usually the fix is the opposite: **filter to the best**. AlpaGasus filtered 52,000 examples down to 9,000 and BEAT the full-data model. Noisy examples actively teach bad behavior. Spend your effort on curation and cleaning, not collection.

---

## 🎯 Concept 1: The Three Data Formats

### The Three Data Formats

Instruction, conversation, preference - each maps to a different fine-tuning goal.

Before you clean or filter anything, you have to shape it. Fine-tuning data comes in three shapes, and the shape is chosen by your GOAL. **Instruction** (system + user + assistant) teaches task-following. **Conversation** (many alternating turns) teaches multi-turn dialogue. **Preference** (prompt + chosen + rejected) teaches which answer is BETTER - the input to DPO. All three are stored as JSONL.

> 📝 **Analogy**
>
> The three formats are three kinds of worksheet. An instruction example is a flashcard (question on front, ideal answer on back). A conversation is a script for a whole scene. A preference pair is an A/B test (“this answer is better than that one”). *Breaks down when:* you pick the worksheet that does not fit the skill - a flashcard (single-turn) cannot teach a model to hold a multi-turn conversation.

You need to fine-tune a model to hold multi-turn support conversations. Which format fits?

- Each format is just a Python dict: instruction/conversation share a `messages` array; preference uses `prompt/chosen/rejected`.
- `to_jsonl` serializes each record to one line - that is what “JSONL” means.
- Notice all three end up as JSONL: the universal on-disk format for every fine-tuning API and framework.

**📝 `01_formats.py`** - *Formats*

In [ ]:
# THE THREE DATA FORMATS - each maps to a different fine-tuning goal.
import json

instruction = {"messages": [
    {"role": "system", "content": "You are a Netsetos support assistant."},
    {"role": "user", "content": "What is the refund policy?"},
    {"role": "assistant", "content": "Full refund within 7 days, 50% from day 8-30, none after 30."}]}

conversation = {"messages": [
    {"role": "user", "content": "What courses do you offer?"},
    {"role": "assistant", "content": "GenAI, Data Science, Cloud, and Full-Stack."},
    {"role": "user", "content": "How much is the GenAI one?"},
    {"role": "assistant", "content": "It is 14,999 rupees with lifetime access."}]}

preference = {"prompt": "What is the refund policy?",
              "chosen": "Full refund within 7 days, 50% from day 8-30, none after 30. Email support@netsetos.com.",
              "rejected": "We have a refund policy. Check the website."}

def to_jsonl(items):
    # JSONL = one JSON object per line: streamable, line-filterable, validated per record.
    return "\n".join(json.dumps(x, ensure_ascii=False) for x in items)

print("INSTRUCTION (system+user+assistant) - one JSONL line:")
print(" ", to_jsonl([instruction])[:96], "...")
print(f"CONVERSATION: {len(conversation['messages'])} alternating messages (multi-turn)")
print(f"PREFERENCE (DPO): keys = {list(preference)}  (prompt + chosen + rejected)")

# Output:
# INSTRUCTION (system+user+assistant) - one JSONL line:
#   {"messages": [{"role": "system", "content": "You are a Netsetos support assistant."}, {"role": " ...
# CONVERSATION: 4 alternating messages (multi-turn)
# PREFERENCE (DPO): keys = ['prompt', 'chosen', 'rejected']  (prompt + chosen + rejected)

#### 💡 What just happened

⚡ What just happened? Three formats, three goals: **instruction** for task-following, **conversation** for multi-turn dialogue, **preference** for DPO alignment. The tradeoff to remember: pick the shape by the SKILL you are teaching, not by what is easiest to collect. And whichever you pick, it ships as JSONL - one record per line, so a 100k-example set streams and filters without ever loading the whole file.

- Toggle instruction / conversation / preference and watch the live JSONL structure change.
- See which fine-tuning goal each shape serves - and why multi-turn needs the conversation format.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Chat Templates - The Silent Killer

### Chat Templates - The Silent Killer

The same messages become a different token string per model. Get it wrong and nothing errors.

This is the cold-open bug, and the most important idea in the lesson. Your clean `messages` array is not what the model trains on - the model trains on a **token string** built by wrapping each turn in that model’s special delimiters. Llama 3, Mistral, Qwen, and Gemma all use different ones. The only safe way to produce the right string is `tokenizer.apply_chat_template(messages, tokenize=False)`, which reads the model’s own Jinja2 template. Hand-format it (or reuse another model’s format) and you get a silent failure.

> 📧 **Analogy**
>
> A chat template is the envelope, not the letter. The same letter (your messages) needs a different envelope for each country’s postal system (each model’s special tokens). Put the letter in the wrong envelope and the post office does not reject it - it just quietly delivers it to the wrong place. *Breaks down when:* you forget that the envelope is added automatically - if you both apply the template AND let the tokenizer add special tokens again, you get two envelopes (duplicated BOS/tokens), which also corrupts training.

You have clean data in Alpaca format. You point a fine-tune at a Llama 3 chat model and hit run. What happens?

- `render` simulates each model’s template so you can SEE the special tokens - in real code you call `apply_chat_template`, never hand-write these.
- The four outputs are the SAME two messages wrapped four different ways (`<|eot_id|>` vs `[/INST]` vs `<|im_end|>` vs `<end_of_turn>`).
- The last block shows Alpaca text fed to Llama 3: none of Llama’s tokens are present - that is the silent failure, in one line.

**📝 `02_template.py`** - *Templates*

In [ ]:
# CHAT TEMPLATES - the same messages become a DIFFERENT token string per model.
# In real code you never hand-write these: tokenizer.apply_chat_template(messages,
# tokenize=False) reads the model's Jinja2 template and emits the exact sequence it
# was trained on. Here we simulate four templates to see why the wrapper matters.
msgs = [{"role": "user", "content": "Refund?"}, {"role": "assistant", "content": "7 days."}]

def render(model, m):
    u, a = m[0]["content"], m[1]["content"]
    if model == "llama-3":
        return (f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{u}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>\n\n{a}<|eot_id|>")
    if model == "mistral":
        return f"<s>[INST] {u} [/INST] {a}</s>"
    if model == "qwen":
        return f"<|im_start|>user\n{u}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>"
    if model == "gemma":
        return f"<start_of_turn>user\n{u}<end_of_turn>\n<start_of_turn>model\n{a}<end_of_turn>"

for mdl in ("llama-3", "mistral", "qwen", "gemma"):
    print(f"  {mdl:9s}: {render(mdl, msgs)!r}")

# The SILENT failure: Alpaca-format text has NONE of Llama-3's special tokens.
alpaca = f"### Instruction:\n{msgs[0]['content']}\n### Response:\n{msgs[1]['content']}"
print("\n  Alpaca text fed to a Llama-3 model (WRONG - no special tokens):")
print("   ", repr(alpaca))
print("  -> trains with no error and no <|eot_id|> stop token = silent garbage.")

# Output:
#   llama-3  : '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nRefund?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n7 days.<|eot_id|>'
#   mistral  : '<s>[INST] Refund? [/INST] 7 days.</s>'
#   qwen     : '<|im_start|>user\nRefund?<|im_end|>\n<|im_start|>assistant\n7 days.<|im_end|>'
#   gemma    : '<start_of_turn>user\nRefund?<end_of_turn>\n<start_of_turn>model\n7 days.<end_of_turn>'
#
#   Alpaca text fed to a Llama-3 model (WRONG - no special tokens):
#     '### Instruction:\nRefund?\n### Response:\n7 days.'
#   -> trains with no error and no <|eot_id|> stop token = silent garbage.

#### 💡 What just happened

⚡ What just happened? The same messages produce four different token strings - and the model only understands its own. Rules that follow: (1) always format with `apply_chat_template()`; (2) after `tokenize=False`, set `add_special_tokens=False` when you tokenize, or BOS/special tokens get **duplicated**; (3) **padding** makes every row in a training batch the same length by appending a filler **PAD** token; never reuse the end-of-sequence token as PAD (`pad_token = eos_token`) or the model can’t tell “filler” from “stop” and truncates early - add a dedicated PAD token; (4) put the padding on the right for training (`padding_side="right"`, so real tokens start at position 0) and on the left for inference (`"left"`, so generation continues from the true end). And the **BOS** (beginning-of-sequence) token you saw as `<|begin_of_text|>` is one of the special tokens the template adds - which is why rule (2) matters. The tradeoff there is none - there is a right answer, and getting it wrong costs you a silent, expensive re-run.

- Pick a model (Llama 3 / Mistral / Qwen / Gemma) and see the same messages rewrapped in its special tokens.
- Flip on “wrong template” to see the Alpaca-on-Llama silent failure - no error, missing stop token.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 📦 **Info**
>
> 🎯 One more formatting rule: train on the completion, not the promptFormatting is not just the template - it is also **which tokens the loss is computed on**. When you tokenize an example into `input_ids` and `labels`, set `labels = -100` on the system + user (prompt) tokens so the loss is measured ONLY on the assistant’s reply (**completion-only training**, or “loss masking”). Otherwise the model spends capacity learning to *predict the questions* instead of answering them - another silent quality drain. Most trainers (TRL’s SFTTrainer, Axolotl) do this for you when you use the chat format, but verify it, especially for multi-turn data.

---

## 🎯 Concept 3: Quality Filtering - Drop Garbage Before Training

### Quality Filtering - Drop Garbage Before Training

Bad data poisons the model. A cheap funnel removes most of it.

Once the data is correctly shaped, you filter it - before it ever reaches the GPU. A few cheap, deterministic checks catch most junk: responses that are too short or too long, exact duplicates, and obvious low-quality patterns (“I don’t know”, repetitive text). Filtering typically removes roughly 20–40% of raw data, and that is a GOOD thing.

> 🧬 **Analogy**
>
> Filtering is airport security for your dataset: a series of cheap checks, fastest first, that stop the obvious problems before the expensive gate (training). A too-short answer, an exact duplicate, or a canned “I don’t know” is confiscated at the first checkpoint. *Breaks down when:* you rely on cheap heuristics alone - they catch the obvious junk but not subtly-wrong answers, which is why the next step adds an LLM judge.

Your quality filter removes about 30% of the raw data. Is that good or bad?

- `DataFilter` runs each example through cheap checks in order: length, then exact-dedup (a hash), then quality heuristics.
- Anything that fails a check is counted in `stats` and skipped - so you get a breakdown of WHY examples were dropped.
- The demo of 5 examples keeps 2: one too short, one low-quality, one exact duplicate are removed.

**📝 `03_filter.py`** - *Filtering*

In [ ]:
# QUALITY FILTER FUNNEL - drop garbage before it poisons the model.
import hashlib
from collections import Counter

class DataFilter:
    def __init__(self, min_w=10, max_w=500):
        self.min_w, self.max_w = min_w, max_w
        self.seen, self.stats = set(), Counter()

    def _resp(self, item):
        for m in reversed(item.get("messages", [])):
            if m["role"] == "assistant":
                return m["content"]
        return ""

    def filter(self, data):
        clean = []
        for item in data:
            self.stats["total"] += 1
            r = self._resp(item)
            wc = len(r.split())
            if wc < self.min_w: self.stats["too_short"] += 1; continue     # length
            if wc > self.max_w: self.stats["too_long"] += 1; continue
            h = hashlib.md5(r.encode()).hexdigest()
            if h in self.seen: self.stats["duplicate"] += 1; continue       # exact dedup
            self.seen.add(h)
            if "I don't know" in r: self.stats["low_quality"] += 1; continue  # heuristic
            uniq = len(set(r.lower().split())) / max(len(r.split()), 1)
            if uniq < 0.3: self.stats["repetitive"] += 1; continue
            self.stats["kept"] += 1; clean.append(item)
        return clean

def mk(u, a): return {"messages": [{"role": "user", "content": u}, {"role": "assistant", "content": a}]}
raw = [
    mk("Refund?", "Full refund within 7 days of purchase, 50% up to day 30, none after that at all."),
    mk("Help", "Ok."),                                                                # too short
    mk("Price?", "I don't know the exact price right now, please check with the support team about it."),  # low quality
    mk("Refund?", "Full refund within 7 days of purchase, 50% up to day 30, none after that at all."),  # duplicate
    mk("Cost?", "The GenAI Complete Course costs 14,999 rupees and includes lifetime access to everything."),
]
f = DataFilter(min_w=10)
clean = f.filter(raw)
print(f"  {len(raw)} raw -> {len(clean)} clean")
print(f"  breakdown: {dict(f.stats)}")

# Output:
#   5 raw -> 2 clean
#   breakdown: {'total': 5, 'kept': 2, 'too_short': 1, 'low_quality': 1, 'duplicate': 1}

#### 💡 What just happened

⚡ What just happened? Five raw examples become two clean ones, with a breakdown of every drop. Quality filtering typically removes roughly 20–40% of raw data - and that is the point: a smaller, cleaner set outperforms a larger noisy one. The tradeoff is recall vs precision on your filters: too aggressive and you lose good edge cases; too lax and noise leaks through. Start cheap and deterministic (length, dedup), then escalate to the LLM judge in the next step for the subtle cases heuristics miss.

- Slide the min/max length thresholds and toggle dedup, and watch examples get kept or dropped live.
- See how the kept count and the drop-reason breakdown shift as you tighten the filters.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Quality Crushes Quantity - The AlpaGasus Filter

### Quality Crushes Quantity - The AlpaGasus Filter

An LLM judge scores each example; you keep only the best. Fewer, better examples win.

Cheap heuristics catch obvious junk; the subtle cases need judgment. The AlpaGasus method (ICLR 2024) uses an **LLM as a judge**: score each example 1–5 on accuracy, helpfulness, and relevance, then keep only the high scorers (roughly the top 15–25%). Filtering 52,000 Alpaca examples to 9,000 this way BEAT the full-data model and trained far faster - for well under ~$100 of scoring.

> 🎓 **Analogy**
>
> It is grading essays instead of counting them. A cheap word-count filter is like checking essays are the right length; the LLM judge is a teacher actually reading each one and keeping the A-grades. *Breaks down when:* your judge is biased or miscalibrated - an LLM judge has the same failure modes you saw in Module 8 (position, verbosity, self-enhancement), so spot-check its scores against a few human ratings before trusting the filter.

You have 52,000 training examples. AlpaGasus says filter. How many should you keep?

- `judge_score` stands in for an LLM judge (deterministic here so it runs offline) - it rates length, specificity, and non-hedging on a 1–5 scale.
- The `KEEP = 4.5` threshold is the AlpaGasus rule: keep only the high scorers.
- On 5 examples, the three concrete/complete answers survive; the vague and the “I don’t know” are dropped.

**📝 `04_score.py`** - *Scoring*

In [ ]:
# QUALITY > QUANTITY (AlpaGasus): score each example, keep only the best.
# A real judge is an LLM rating 1-5 on accuracy/helpfulness/relevance. Here we use a
# deterministic stand-in so the demo runs offline; the RULE is what matters.
def judge_score(resp):
    words = resp.split(); n = len(words)
    length_ok = 1.0 if 8 <= n <= 120 else (0.4 if n < 8 else 0.7)
    specific = 1.0 if any(c.isdigit() for c in resp) else 0.6       # concrete facts score higher
    hedge = 0.3 if ("I don't know" in resp or "check the website" in resp.lower()) else 1.0
    return round(5 * (0.4*length_ok + 0.3*specific + 0.3*hedge), 2)  # 1-5 scale

pool = [
    "Full refund within 7 days, 50% from day 8 to 30, none after 30.",
    "We have a refund policy, please check the website for details.",
    "The GenAI course costs 14,999 rupees with lifetime access and GCP credits.",
    "I don't know, sorry.",
    "Live classes run daily at 7 PM IST; recordings are posted within 2 hours.",
]
KEEP = 4.5
scored = [(judge_score(r), r) for r in pool]
kept = [r for s, r in scored if s >= KEEP]
for s, r in sorted(scored, reverse=True):
    print(f"  {s:.2f}  {'KEEP' if s >= KEEP else 'drop'}  {r[:52]}")
print(f"  AlpaGasus rule: keep score >= {KEEP} -> {len(kept)}/{len(pool)} ({round(100*len(kept)/len(pool))}%) survive")

# Output:
#   5.00  KEEP  The GenAI course costs 14,999 rupees with lifetime a
#   5.00  KEEP  Live classes run daily at 7 PM IST; recordings are p
#   5.00  KEEP  Full refund within 7 days, 50% from day 8 to 30, non
#   3.35  drop  We have a refund policy, please check the website fo
#   2.15  drop  I don't know, sorry.
#   AlpaGasus rule: keep score >= 4.5 -> 3/5 (60%) survive

#### 💡 What just happened

⚡ What just happened? The judge keeps the specific, complete answers and drops the vague ones - 3 of 5 survive at threshold 4.5. (This 5-example demo is deliberately mostly high-quality, so 3 of 5 survive; on a real 52k set full of mediocre data, that same 4.5 threshold keeps only roughly 15–25%.) This is the highest-ROI step in data prep: a scoring pass that costs a few dollars (a cheap 4o-mini-class judge on tens of thousands of rows) can filter 52k to 9k and IMPROVE the model. The tradeoff is judge cost and bias vs signal: you pay a little per example and must calibrate the judge, but you buy a smaller, better, faster-training dataset. One caution the filters share: aggressive length/quality/dedup filtering is *not* distribution-neutral - it can quietly drop a whole task type or class, so check the per-task keep rate after filtering and re-balance if one category collapsed.

- Slide the score threshold and watch the kept percentage fall while the projected downstream score rises.
- Find the sweet spot where a small, high-quality set beats the full noisy one.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Synthetic Data Generation - and Always Filter It

### Synthetic Data Generation - and Always Filter It

No data? Have a strong model generate it. Then treat it like any other raw data: filter.

Sometimes you do not have enough data. The modern fix is to **generate it**: prompt a strong model with your knowledge base and seed patterns to produce instruction or preference examples. This is mainstream and legitimate (Microsoft’s Phi models leaned heavily on synthetic data) - with one rule: synthetic data is *raw* data. It goes through the exact same cleaning and quality filter as anything you collected.

> 🤖 **Analogy**
>
> Synthetic data is a teaching assistant writing practice problems. It scales your dataset cheaply - but you still proofread every problem before handing it to students. *Breaks down when:* you trust the generator blindly - ungrounded synthetic data can be repetitive, wrong, or leak the generator’s own quirks, so it must be deduped and quality-scored just like collected data.

A big model generates 5,000 training examples for you. Do you train on them directly?

- This uses the CURRENT Google SDK: `from google import genai; client = genai.Client()` with `gemini-2.5-flash` (the old `google.generativeai` + `gemini-2.0-flash` is retired).
- It prompts the model with a knowledge base and parses a JSON array into the messages format from step 1.
- The final line is the whole point: whatever comes back, dedup and quality-score it (steps 3–4) before training.

**📝 `05_synthetic.py`** - *google-genai*

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


In [ ]:
# SYNTHETIC DATA - use a strong model to generate training examples, then FILTER.
# Requires GOOGLE_API_KEY and `pip install google-genai` (see the two cells above).
# Guarded so Restart & Run All is safe without a key; set the key to run it live.
if os.environ.get("GOOGLE_API_KEY"):
    from google import genai
    from google.genai import types

    client = genai.Client()   # reads GOOGLE_API_KEY

    KB = ("Netsetos GenAI course: 14,999 rupees, lifetime access. "
          "Refund: full within 7 days, 50% day 8-30, none after 30. Live classes 7 PM IST daily.")

    def generate_instruction(n: int = 5) -> list[dict]:
        prompt = (f"Generate {n} diverse user questions about this knowledge base plus ideal "
                  f'assistant answers. Return a JSON array of {{"user": "...", "assistant": "..."}}.\n'
                  f"Knowledge:\n{KB}\nJSON:")
        txt = client.models.generate_content(
            model="gemini-2.5-flash", contents=prompt,
            config=types.GenerateContentConfig(temperature=0.7)).text
        import json, re
        txt = re.sub(r"```(json)?", "", txt).strip()
        pairs = json.loads(txt)
        return [{"messages": [{"role": "system", "content": "You are a Netsetos support assistant."},
                              {"role": "user", "content": p["user"]},
                              {"role": "assistant", "content": p["assistant"]}]} for p in pairs]

    examples = generate_instruction(n=5)
    print(f"Generated {len(examples)} synthetic instruction examples. First:")
    print(" ", examples[0]["messages"][1]["content"])
    print("  ALWAYS filter synthetic data (dedup + quality score) before training on it.")
else:
    print("Set GOOGLE_API_KEY (and pip install google-genai) to run the live synthetic-data demo.")

# Output: (varies by run - synthetic generation is non-deterministic)
# Generated 5 synthetic instruction examples. First:
#   How long do I have to get a full refund on the GenAI course?
#   ALWAYS filter synthetic data (dedup + quality score) before training on it.

#### 💡 What just happened

⚡ What just happened? A strong model turns a small knowledge base into training examples - but the load-bearing line is the last one: **always filter synthetic data**. The tradeoff is scale vs trust: generation is cheap and fast, but ungrounded output drifts, repeats, and inherits the generator’s biases, so it must survive the same dedup + quality gate as collected data. Tools like Distilabel automate exactly this generate-then-judge loop at scale.

---

## 🎯 Concept 6: The Cleaning Pipeline - Cheapest Filters First

### The Cleaning Pipeline - Cheapest Filters First

Dedup, contamination, PII - run the free checks before the expensive ones.

Beyond quality lies cleanliness. A production pipeline runs a sequence of filters ordered by cost: the free ones first, the LLM-based ones last. Exact dedup (a hash) is free; near-duplicate detection (MinHash/LSH) catches reformulations; PII removal ([Microsoft Presidio](https://github.com/microsoft/presidio)) protects users and keeps you compliant; a contamination check makes sure your training set does not secretly contain the benchmark you plan to evaluate on.

> 🏯 **Analogy**
>
> It is a recycling sort line. Cheap magnets and screens pull out the obvious stuff first (exact duplicates, wrong-length); the expensive hand-sort (an LLM judge) comes last, only on what survived. Running it in the other order - hand-sorting everything, then discovering half were duplicates - wastes the expensive step. *Breaks down when:* you skip the contamination screen - benchmark leakage silently inflates your eval scores and hides a model that only memorized the test.

Your training data contains Aadhaar numbers and email addresses. What do you do?

- Stage 1 is exact dedup via a hash (free). Stage 2 is near-dup: `jaccard` over word-shingles drops reformulations a hash misses (real code uses MinHash/LSH via datasketch).
- Stage 3 is a **contamination** check: `contaminated` drops any doc that shares a 4-gram with a held-out benchmark, so eval questions never leak into training.
- Stage 4 is PII: `scrub` REPLACES Aadhaar/email/phone with typed placeholders (never deletes). The demo: 4 docs -> one near-duplicate and one benchmark-leak dropped -> survivors PII-clean.

**📝 `06_clean.py`** - *Cleaning*

In [ ]:
# THE CLEANING PIPELINE - run the cheapest filters first.
import hashlib, re

def shingles(text, k=3):
    w = text.lower().split()
    return set(tuple(w[i:i+k]) for i in range(max(len(w)-k+1, 1)))

def jaccard(a, b):
    A, B = shingles(a), shingles(b)
    return len(A & B) / max(len(A | B), 1)

# a tiny held-out benchmark we must NEVER train on (contamination = benchmark leakage)
BENCHMARK = ["what is the capital of france", "who wrote pride and prejudice"]
def contaminated(text, bench, n=4):
    tg = shingles(text, n)
    return any(shingles(b, n) & tg for b in bench)   # shared 4-gram = leaked eval question

PII = [(re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b"), "<AADHAAR>"),
       (re.compile(r"\b[\w.]+@[\w.]+\.\w+\b"), "<EMAIL>"),
       (re.compile(r"\b(?:\+91[- ]?)?[6-9]\d{9}\b"), "<PHONE>")]
def scrub(text):
    for pat, tag in PII:
        text = pat.sub(tag, text)   # REPLACE with a placeholder (not delete) to keep structure
    return text

docs = [
    "You can get a full refund within 7 days of purchase. Contact rahul@netsetos.com or call 9876543210 for help.",
    "You can get a full refund within 7 days of purchase. Reach rahul@netsetos.com or call 9876543210 for help.",   # near-duplicate
    "My Aadhaar number is 1234 5678 9012 and I would like to request a refund on my order today.",
    "Quick quiz to warm up: what is the capital of france? Great, now back to your refund question.",              # contaminated
]
seen, exact = set(), []
for d in docs:                                   # 1) exact dedup (free)
    h = hashlib.md5(d.encode()).hexdigest()
    if h not in seen: seen.add(h); exact.append(d)
near = []
for d in exact:                                  # 2) near-dup (cheap) - real code uses MinHash/LSH
    if all(jaccard(d, k) <= 0.6 for k in near): # via datasketch with a tuned Jaccard threshold
        near.append(d)
clean = [d for d in near if not contaminated(d, BENCHMARK)]   # 3) drop benchmark leakage
final = [scrub(d) for d in clean]                # 4) PII placeholder-replace (before any LLM step)
print(f"  {len(docs)} docs -> exact {len(exact)} -> near-dedup {len(near)} -> "
      f"decontaminated {len(clean)} -> PII-scrubbed:")
for d in final:
    print("   ", d)

# Output:
#   4 docs -> exact 4 -> near-dedup 3 -> decontaminated 2 -> PII-scrubbed:
#     You can get a full refund within 7 days of purchase. Contact <EMAIL> or call <PHONE> for help.
#     My Aadhaar number is <AADHAAR> and I would like to request a refund on my order today.

#### 💡 What just happened

⚡ What just happened? Four docs become two - the near-duplicate a hash would miss is caught by shingle overlap, a doc that leaked a benchmark question is dropped by the contamination check, and personal data is replaced with placeholders that keep the sentence trainable. The ordering is the lesson: run the free filters (length, exact dedup) before the paid LLM-scoring filter, and always run the contamination check (n-gram overlap vs MMLU/GSM8K/HumanEval) - benchmark leakage inflates eval scores and hides a model that only memorized the test. The tradeoff is thoroughness vs cost, which the cheapest-first order optimizes directly.

- Toggle each stage (length, dedup, PII, contamination, LLM score) on or off and watch the running cost and what each stage catches.
- See why the cheapest filters must run before the expensive LLM judge.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Splits, Versioning & Tools

### Splits, Versioning & Tools

Stratified 80/10/10, a real validation set, and version everything.

Clean data still needs to be split and tracked. Split **80/10/10** (train/val/test) for small sets, and **stratify by task type** so every category is represented in each split. Keep at least 50–100 validation examples or your loss curve is noise. Then version it: **DVC** git-tracks large files via tiny metafiles, and `push_to_hub()` shares it with a dataset card. The modern tooling stack: Distilabel generates, Lilac explores, Argilla annotates.

> 🎲 **Analogy**
>
> Stratified splitting is dealing a deck so each player gets the same mix of suits. Deal randomly from a sorted deck and one player gets all the hearts (one task type) - your validation set no longer represents the data. *Breaks down when:* your validation set is tiny - fewer than ~50 examples and the loss curve wobbles so much you cannot tell a good checkpoint from a bad one.

You split a task-sorted dataset 80/10/10 without stratifying. What goes wrong?

- `stratified_split` groups rows by `task`, then splits EACH group 80/10/10 and concatenates - so every task keeps its proportion in every split.
- The alternative (a plain slice of sorted data) would put whole tasks into one split.
- The output confirms train keeps all three task types in equal measure.

**📝 `07_split.py`** - *Splitting*

In [ ]:
# STRATIFIED SPLIT - keep every task type proportional across train/val/test.
from collections import Counter

data = ([{"task": "refund", "id": i} for i in range(10)] +
        [{"task": "pricing", "id": i} for i in range(10)] +
        [{"task": "schedule", "id": i} for i in range(10)])

def stratified_split(rows, ratios=(0.8, 0.1, 0.1)):
    by = {}
    for r in rows:
        by.setdefault(r["task"], []).append(r)
    train, val, test = [], [], []
    for task, items in sorted(by.items()):      # per-task, so proportions are preserved
        n = len(items)
        n_tr, n_va = round(n*ratios[0]), round(n*ratios[1])
        train += items[:n_tr]; val += items[n_tr:n_tr+n_va]; test += items[n_tr+n_va:]
    return train, val, test

tr, va, te = stratified_split(data)
print(f"  {len(data)} examples -> train {len(tr)} / val {len(va)} / test {len(te)}")
print(f"  train task mix: {dict(Counter(r['task'] for r in tr))}  (each task kept proportional)")

# Output:
#   30 examples -> train 24 / val 3 / test 3
#   train task mix: {'pricing': 8, 'refund': 8, 'schedule': 8}  (each task kept proportional)

#### 💡 What just happened

⚡ What just happened? The stratified split keeps every task type proportional across train/val/test - so your validation loss actually reflects the data. Beyond the split: keep ≥50–100 validation examples, version the dataset with DVC (metafiles in Git, data in cloud storage), and log every filter and threshold to a machine-readable file so a run is reproducible. The tradeoff in split ratios is train signal vs eval reliability: more validation data gives steadier curves but less to train on - 80/10/10 is the small-data default, shifting to 98/1/1 at millions of examples.

> ✅ **Info**
>
> 💾 Versioning in three commandsOnce split, version it so a run is reproducible: `dvc add data/train.jsonl` writes a tiny `.dvc` metafile holding the data’s hash → `git commit` that metafile → `dvc push` sends the actual data to S3/GCS. To share, `dataset.push_to_hub("org/name", private=True)` with a dataset card documenting the filters and thresholds you applied. Git tracks only the pointer and the recipe; the data lives in cloud storage.

---

## 🎯 Concept 8: India Data Preparation

### India Data Preparation

Tokenizer fertility, NFC normalization, Hinglish, AI4Bharat, and DPDP.

Indian-language data adds three twists. First, **tokenizer fertility**: a generic tokenizer may split Hindi into 4–8 tokens per word (Telugu even more), which truncates training examples and multiplies inference cost - so you choose the base model by fertility first. Second, **normalization**: always apply Unicode NFC (and strip zero-width characters) before tokenizing, or identical-looking text hashes and tokenizes differently. Third, **Hinglish** code-mixing has no standard spelling. The AI4Bharat ecosystem (Sangraha, IndicCorp v2, BPCC, IndicAlign) supplies open, DPDP-friendly data.

> 📏 **Analogy**
>
> Fertility is how many syllables a reader needs to sound out a word. A generic tokenizer stammers through Hindi (many tokens per word); an Indic-first tokenizer reads it fluently (few). NFC normalization is spell-checking the encoding so “the same word” is literally the same bytes. *Breaks down when:* you assume English intuitions - a 4k context that holds ~3,000 English words holds only a few hundred Hindi words on a high-fertility tokenizer.

Llama 3 tokenizes your Hindi at ~6 tokens/word. What is the impact on fine-tuning?

- `FERT` holds illustrative tokens-per-word by model and language; the loop shows how many words fit a 4k context for each.
- The gap is the point: Sarvam-1 fits several times more Hindi/Telugu per context than Llama 3.
- The NFC line shows normalization collapsing a combining-accent sequence to one code point - always do this before tokenizing.

**📝 `08_india.py`** - *India*

In [ ]:
# INDIA - tokenizer fertility decides viability, and NFC-normalize before tokenizing.
import unicodedata

# tokens-per-word by (model, language) - illustrative from AI4Bharat / Sarvam reporting.
FERT = {"llama-3": {"english": 1.3, "hindi": 6.0, "telugu": 11.0},
        "sarvam-1": {"english": 1.5, "hindi": 1.8, "telugu": 2.0}}
CTX = 4096
for lang in ("english", "hindi", "telugu"):
    la, sa = FERT["llama-3"][lang], FERT["sarvam-1"][lang]
    print(f"  {lang:8s}: llama-3 ~{la:4.1f} tok/word ({int(CTX/la):4d} words fit a {CTX}-ctx) | "
          f"sarvam-1 ~{sa:3.1f} tok/word ({int(CTX/sa):4d} words)")
print("  -> On Hindi/Telugu, Sarvam-1's Indic tokenizer fits 3-5x more words per context than Llama-3;")
print("     on English the two are comparable. Choose the base model by fertility on YOUR language.")

decomposed = "café"   # combining acute accent - looks identical to an e-acute
nfc = unicodedata.normalize("NFC", decomposed)
print(f"  NFC: {len(decomposed)} code points -> {len(nfc)} (combining marks collapse to one).")

# Output:
#   english : llama-3 ~ 1.3 tok/word (3150 words fit a 4096-ctx) | sarvam-1 ~1.5 tok/word (2730 words)
#   hindi   : llama-3 ~ 6.0 tok/word ( 682 words fit a 4096-ctx) | sarvam-1 ~1.8 tok/word (2275 words)
#   telugu  : llama-3 ~11.0 tok/word ( 372 words fit a 4096-ctx) | sarvam-1 ~2.0 tok/word (2048 words)
#   -> On Hindi/Telugu, Sarvam-1's Indic tokenizer fits 3-5x more words per context than Llama-3;
#      on English the two are comparable. Choose the base model by fertility on YOUR language.
#   NFC: 5 code points -> 4 (combining marks collapse to one).

> 📦 **Info**
>
> 🇮🇳 The India data stack (Jun 2026)
> - **AI4Bharat:** Sangraha (251B tokens, 22 languages), IndicCorp v2 (CC-0), BPCC (parallel sentences), IndicAlign-Instruct (ready-made SFT data), IndicLLMSuite (the dataset blueprint, [github.com/AI4Bharat/IndicLLMSuite](https://github.com/AI4Bharat/IndicLLMSuite)). Bhashini for government data.
> - **Hinglish:** no standard orthography (“kya / kia / kyaa”); needs token-level language ID. COMI-LINGUA and HinglishEval are benchmarks.
> - **Normalize first:** Unicode NFC, strip zero-width characters, then language-specific normalization (the `indicnlp` library).
> - **Fertility:** choose the base model by tokens/word on YOUR language - Sarvam-1 (~1.4–2.1) uses roughly 3–5× fewer tokens on Hindi/Telugu than a generic tokenizer (2–4× across Indic broadly).
> - **DPDP Act 2023:** consent-centric (no GDPR-style “legitimate interest”), penalties up to ₹250 crore, full enforcement expected from ~May 2027. Prefer CC-0 data (IndicCorp v2) + synthetic + Presidio PII removal.

#### 💡 What just happened

⚡ What just happened? India data prep is three disciplines at once: pick the base model by **fertility**, **NFC-normalize** before tokenizing, and source from open **AI4Bharat** data to stay DPDP-clean. The tradeoff that decides your base model is fertility over familiarity: a smaller Indic-first model beats a larger generic one on Hindi because it fits more content per token - and code-mixed Hinglish remains the hardest case even then.

- Pick a language and a model and watch tokens/word, how many words fit a context window, and the cost multiplier.
- See why a smaller Indic-first model beats a larger generic one on Hindi and Telugu.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The data-prep pipeline in one path
> - **Shape it.** Pick the format by goal (instruction / conversation / preference); store as JSONL.
> - **Wrap it right.** Format with `apply_chat_template()` - the one step that silently ruins runs.
> - **Filter for quality.** Cheap heuristics, then an LLM judge (AlpaGasus): keep the best ~15–25% (roughly a fifth).
> - **Generate + clean.** Synthetic data if needed (then filter it); dedup, contamination check, PII replace - cheapest first.
> - **Split + version.** Stratified 80/10/10, ≥50–100 val, DVC + a dataset card. For Indic: fertility + NFC + DPDP.
> The through-line: **data prep is about 80% of fine-tuning, and quality - of both content and FORMAT - beats quantity every time.**

> 📦 **Info**
>
> 🔗 Where this goes next
> - In Lesson 9.3 we’ll take this JSONL into **Vertex AI** managed fine-tuning.
> - Lesson 9.4 picks this up for **open-source** training (Axolotl / Unsloth / TRL), which consume exactly these files.
> - In Lesson 9.5 we’ll come back to the **LoRA & QLoRA** training loop that learns from this data.
> - Later, in Module 14 (**LLMOps**), we’ll return to versioning and monitoring datasets in production.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Data Preparationfor Fine-Tuning**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-9_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-9.2.html` - regenerate this notebook whenever the source HTML is updated.*
